In [ ]:
import numpy as np
from scipy.io import loadmat
import h5py
import matplotlib.pyplot as plt
from scipy.ndimage import zoom
import torch

In [ ]:
raw_path = {
    "source": "Helmholtz.h5",
}
save_path = {
    "train": "trainset.h5",
    "test": "testset.h5",
}

In [ ]:
def resize_batch_numpy(data, target_size=64, is_binary=False):
    """
    data: numpy array with shape [B, 1, H, W]
    target_size: int (e.g., 64)
    is_binary: True -> nearest; False -> bilinear
    """
    B, C, H, W = data.shape
    scale = target_size / H
    order = 0 if is_binary else 1  # 0=nearest, 1=bilinear

    # 插值  -> 注意 zoom 可以对所有维度一起缩放！
    data_resized = zoom(data, (1, 1, scale, scale), order=order)

    return data_resized

In [ ]:
def minmax_norm(x, xmin, xmax):
    """Min-Max to [-1, 1]"""
    return 2 * (x - xmin) / (xmax - xmin) - 1

In [ ]:
def obs_field_reconstruction(obs_batch, field_size):
    B, N, T = obs_batch.shape
    width = T - 2

    obs_recon_batch = torch.zeros((B, width, field_size, field_size),
                                 device=obs_batch.device)

    x_pos = ((obs_batch[..., 0] + 1) / 2 * field_size).long()  # (B, N)
    y_pos = ((obs_batch[..., 1] + 1) / 2 * field_size).long()  # (B, N)

    for b in range(B):
        for w in range(width):
            obs_recon_batch[b, w, x_pos[b], y_pos[b]] = obs_batch[b, :, w + 2]

    return obs_recon_batch

### Trainset Process

In [ ]:
obs_label = np.empty([5200, 1, 128, 128])
target = np.empty([5200, 1, 128, 128])

with h5py.File(raw_path["source"], "r") as f:
    '''
    a = PDE parameters / coefficient field (Target for Inversion)
    Represents coefficients / material properties / medium parameters in the PDE, such as:
        diffusion coefficient
        conductivity
        wave velocity
        permeability

    bc = Boundary conditions; single numerical scalar [()]

    u = Solution from PDE forward modeling; observed values in inversion (full observation)
    '''

    val_count = 0
    for val in f.keys():
        if val_count < 5200:
            target_point = f[val]["a"]
            obs_point = f[val]["u"]

            target[val_count][0] = target_point
            obs_label[val_count][0] = obs_point

            val_count += 1


obs_label = resize_batch_numpy(obs_label, target_size=64, is_binary=False)
target = resize_batch_numpy(target, target_size=64, is_binary=True)
print(target.shape,obs_label.shape)

In [ ]:
target_trainset = target[0:5000]
target_testset = target[5000:]
obs_label_trainset = obs_label[0:5000]
obs_label_testset = obs_label[5000:]

In [ ]:
target_max, target_min = target_trainset.max(), target_trainset.min()
target_trainset_norm = minmax_norm(target_trainset, target_min, target_max)
obs_max, obs_min = obs_label_trainset.max(), obs_label_trainset.min()
obs_label_train_norm = minmax_norm(obs_label_trainset, obs_min, obs_max)

In [ ]:
obs_set_train = []

for sample in obs_label_train_norm:
    set_list = []
    for x in range(obs_label_train_norm.shape[-1]):
        for y in range(obs_label_train_norm.shape[-1]):
            element = [minmax_norm(x,0,64),minmax_norm(y,0,64)] + sample[:,x,y].tolist()
            set_list.append(element)
    obs_set_train.append(set_list)
obs_set_train = np.array(obs_set_train)

### Testset Process

In [ ]:
# target_max, target_min = target_trainset.max(), target_trainset.min()
target_test_norm = minmax_norm(target_testset, target_min, target_max)
# obs_max, obs_min = obs_label_trainset.max(), obs_label_trainset.min()
obs_label_testset_norm = minmax_norm(obs_label_testset, obs_min, obs_max)

In [ ]:
obs_set_test = []

for sample in obs_label_testset_norm:
    set_list = []
    for x in range(obs_label_testset_norm.shape[-1]):
        for y in range(obs_label_testset_norm.shape[-1]):
            element = [minmax_norm(x,0,64),minmax_norm(y,0,64)] + sample[:,x,y].tolist()
            set_list.append(element)
    obs_set_test.append(set_list)
obs_set_test = np.array(obs_set_test)

In [ ]:
with h5py.File(f'../{save_path["train"]}', "w") as f:
    f.create_dataset("target", data=target_trainset_norm.squeeze(1))
    f.create_dataset("obs", data=obs_set_train)
    f.create_dataset("obs_label", data=obs_label_train_norm)
print(target_trainset_norm.squeeze(1).shape,obs_set_train.shape,obs_label_train_norm.shape)

In [ ]:
with h5py.File(f'../{save_path["test"]}', "w") as f:
    f.create_dataset("target", data=target_test_norm.squeeze(1))
    f.create_dataset("obs", data=obs_set_test)
    f.create_dataset("obs_label", data=obs_label_testset_norm)
    print(target_test_norm.squeeze(1).shape,obs_set_test.shape,obs_label_testset_norm.shape)